In [1]:
from copy import deepcopy
import torch
import sys
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda import amp
from spikingjelly.activation_based import functional, surrogate, neuron, layer
from spikingjelly.activation_based.model import parametric_lif_net
from spikingjelly.datasets.dvs128_gesture import DVS128Gesture
from torch.utils.data import DataLoader
import time
import os
import argparse
import datetime

In [2]:
torch.manual_seed(1)

In [3]:
T = 16
b = 4
j = 8
lr = 0.001
epochs = 20
channels = 128

data_dir = os.path.expanduser('~/datasets/DVSGesture/')

In [4]:
device = 'cuda:0'

In [5]:
class DVSGestureNet(nn.Module):
    def __init__(self, channels=128, spiking_neuron: callable=None, is_seperable=False, kernel_size=3, **kwargs):
        super().__init__()

        conv = []
        ## First Layer
        conv.append(layer.Conv2d(2, channels, kernel_size=kernel_size, 
                                         padding=1, bias=False))
        conv.append(layer.BatchNorm2d(channels, step_mode='m'))

        ## Middle Layers
        for i in range(4):
            if is_seperable:
                conv.append(layer.Conv2d(channels, channels,
                                         kernel_size=kernel_size, groups=in_channels,
                                         padding=1, bias=False)
                )
                conv.append(layer.BatchNorm2d(channels))
                conv.append(layer.Conv2d(channels, channels, kernel_size=1))
            else:
                conv.append(layer.Conv2d(channels, channels, kernel_size=kernel_size, 
                                         padding=1, bias=False))
                conv.append(layer.BatchNorm2d(channels))
                
            conv.append(spiking_neuron(**deepcopy(kwargs)))


        self.conv_fc = nn.Sequential(
            *conv,

            layer.AdaptiveAvgPool2d(output_size=(1, 1)),
            layer.Linear(in_features=channels, out_features=11)
        )

    def forward(self, x: torch.Tensor):
        return self.conv_fc(x)


In [6]:
train_set = DVS128Gesture(root=data_dir, train=True, data_type='frame', frames_number=T, split_by='number')
test_set = DVS128Gesture(root=data_dir, train=False, data_type='frame', frames_number=T, split_by='number')

The directory [/home/tahaf/datasets/DVSGesture/frames_number_16_split_by_number] already exists.
The directory [/home/tahaf/datasets/DVSGesture/frames_number_16_split_by_number] already exists.


In [7]:
train_data_loader = torch.utils.data.DataLoader(
    dataset=train_set,
    batch_size=b,
    shuffle=True,
    drop_last=True,
    num_workers=j,
    pin_memory=True
)

test_data_loader = torch.utils.data.DataLoader(
    dataset=test_set,
    batch_size=b,
    shuffle=True,
    drop_last=False,
    num_workers=j,
    pin_memory=True
)

In [8]:
scaler = amp.GradScaler()

In [9]:
def check_model(net):
    start_time = time.time()
    max_test_acc = -1
    for epoch in range(epochs):
        net.train()
        train_loss = 0
        train_acc = 0
        train_samples = 0
        for frame, label in train_data_loader:
            optimizer.zero_grad()
            frame = frame.to(device)
            frame = frame.transpose(0, 1)  # [N, T, C, H, W] -> [T, N, C, H, W]
            label = label.to(device)
            label_onehot = F.one_hot(label, 11).float()
    
            if scaler is not None:
                with amp.autocast():
                    out_fr = net(frame).mean(0)
                    loss = F.mse_loss(out_fr, label_onehot)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out_fr = net(frame).mean(0)
                loss = F.mse_loss(out_fr, label_onehot)
                loss.backward()
                optimizer.step()
    
            train_samples += label.numel()
            train_loss += loss.item() * label.numel()
            train_acc += (out_fr.argmax(1) == label).float().sum().item()
    
            functional.reset_net(net)
    
        train_loss /= train_samples
        train_acc /= train_samples
    
        lr_scheduler.step()
    
        net.eval()
        test_loss = 0
        test_acc = 0
        test_samples = 0
        with torch.no_grad():
            for frame, label in test_data_loader:
                frame = frame.to(device)
                frame = frame.transpose(0, 1)  # [N, T, C, H, W] -> [T, N, C, H, W]
                label = label.to(device)
                label_onehot = F.one_hot(label, 11).float()
                out_fr = net(frame).mean(0)
                loss = F.mse_loss(out_fr, label_onehot)
                test_samples += label.numel()
                test_loss += loss.item() * label.numel()
                test_acc += (out_fr.argmax(1) == label).float().sum().item()
                functional.reset_net(net)
        test_loss /= test_samples
        test_acc /= test_samples
        max_test_acc = max(max_test_acc, test_acc)
        
        print(f'epoch = {epoch}, train_loss ={train_loss: .4f}, train_acc ={train_acc: .4f}, test_loss ={test_loss: .4f}, test_acc ={test_acc: .4f}, max_test_acc ={max_test_acc: .4f}')
    
    end_time = time.time()
    print(f'total time: {end_time - start_time}')

In [10]:
net = DVSGestureNet(
    channels=channels,
    spiking_neuron=neuron.LIFNode,
    surrogate_function=surrogate.ATan(),
    is_seperable=False,
    detach_reset=True
)

In [11]:
net.to(device)

DVSGestureNet(
  (conv_fc): Sequential(
    (0): Conv2d(2, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, step_mode=s)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, step_mode=m)
    (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, step_mode=s)
    (3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, step_mode=s)
    (4): LIFNode(
      v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=s, backend=torch, tau=2.0
      (surrogate_function): ATan(alpha=2.0, spiking=True)
    )
    (5): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, step_mode=s)
    (6): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, step_mode=s)
    (7): LIFNode(
      v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=s, backend=torch, tau=2.0
      (surrogate_function): ATan(alpha=2.0, spiking=True)
    )
 

In [12]:
sum(p.numel() for p in net.parameters())

594827

In [13]:
functional.set_step_mode(net, step_mode='m')

In [14]:
optimizer = torch.optim.Adam(net.parameters(), lr=lr)

In [15]:
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)

In [16]:
check_model(net)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (8192x1 and 128x11)

In [ ]:
net = DVSGestureNet(
    channels=channels,
    spiking_neuron=neuron.LIFNode,
    surrogate_function=surrogate.ATan(),
    is_seperable=True,
    detach_reset=True
)

In [ ]:
net.to(device)

In [ ]:
optimizer = torch.optim.Adam(net.parameters(), lr=lr)

In [ ]:
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)

In [ ]:
check_model(net)

In [ ]:
sum(p.numel() for p in net.parameters())